### Match Schedule Scraping

In [1]:
import streamlit as st
import pandas as pd
import soccerdata as sd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Pitch
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from PIL import Image 
import requests
import os

print("Packages imported successfully")

[08/22/26 21:04:58] INFO     No custom team name replacements found. You can configure these in       _config.py:92
                             C:\Users\rumja\soccerdata\config\teamname_replacements.json.                          

[08/22/26 21:04:59] INFO     No custom league dict found. You can configure additional leagues in    _config.py:190
                             C:\Users\rumja\soccerdata\config\league_dict.json.                                    

Packages imported successfully


In [2]:
raw_file = "data/raw/world_cup_matches_raw.csv"

In [3]:
ws = sd.WhoScored(
    leagues="INT-World Cup",
    seasons=2026
)

[08/22/26 21:06:00] INFO     Saving cached data to C:\Users\rumja\soccerdata\data\WhoScored          _common.py:250

[08/22/26 21:08:55] ERROR                                                                            _common.py:567
                                             The ChromeDriver was unable to initiate/spawn a new                   
                                             WebBrowser. You will not be able to scrape new data.                  
                                             Message: session not created: cannot connect to chrome                
                             at 127.0.0.1:9222                                                                     
                             from chrome not reachable; For documentation on this error, please                    
                             visit:                                                                                
                             https://www.selenium.dev/documentation/webdriver/troubleshooting/errors               
                             #sessionnotcreatedexception                                                           
                             Stacktrace:                                                                           
                                     GetHandleVerifier [0x0x7ff699d0cda5+78885]                                    
                                     GetHandleVerifier [0x0x7ff699d0ce00+78976]                                    
                                     (No symbol) [0x0x7ff699ac99fc]                                                
                                     (No symbol) [0x0x7ff699aba831]                                                
                                     (No symbol) [0x0x7ff699b0d252]                                                
                                     (No symbol) [0x0x7ff699b0246e]                                                
                                     (No symbol) [0x0x7ff699b569ee]                                                
                                     (No symbol) [0x0x7ff699b56180]                                                
                                     (No symbol) [0x0x7ff699b48963]                                                
                                     (No symbol) [0x0x7ff699b116b1]                                                
                                     (No symbol) [0x0x7ff699b12443]                                                
                                     GetHandleVerifier [0x0x7ff699fe4eed+3061101]                                  
                                     GetHandleVerifier [0x0x7ff699fdf33d+3037629]                                  
                                     GetHandleVerifier [0x0x7ff699ffe592+3165202]                                  
                                     GetHandleVerifier [0x0x7ff699d2730e+186766]                                   
                                     GetHandleVerifier [0x0x7ff699d2eb3f+217535]                                   
                                     GetHandleVerifier [0x0x7ff699d159b4+114740]                                   
                                     GetHandleVerifier [0x0x7ff699d15b69+115177]                                   
                                     GetHandleVerifier [0x0x7ff699cfc368+10728]                                    
                                     BaseThreadInitThunk [0x0x7ff9fbd07374+20]                                     
                                     RtlUserThreadStart [0x0x7ff9fbf9cc91+33]                                      
                                                                                                                   
                                                                                                                   

In [4]:
def get_matches(stage_id, year, month):
    url = f"https://www.whoscored.com/tournaments/{stage_id}/data/?d={year}{month:02d}"

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    return pd.DataFrame(data['tournaments'][0]['matches'])                  

In [5]:
stages = ws.read_season_stages()
stage_ids = stages['stage_id'].unique().tolist()
stage_ids = [int(stage_id) for stage_id in stage_ids]

In [6]:
if os.path.exists(raw_file):
    world_cup_matches = pd.read_csv(raw_file)

else:
    all_matches = []

    for stage_id in stage_ids:
        if stage_id == 23752:
            all_matches.append(get_matches(stage_id, 2026, 6))
            all_matches.append(get_matches(stage_id, 2026, 7))
        else:
            all_matches.append(get_matches(stage_id, 2026, 6))

    world_cup_matches = pd.concat(all_matches, ignore_index=True)
    world_cup_matches.to_csv(raw_file, index=False)

### Event Scraping

In [10]:
match_ids = world_cup_matches['id'].unique().tolist()

In [16]:
events_list = []